**Code for Data Cleaning:**

In [3]:
import pandas as pd
import numpy as np
import kagglehub
import os

# Load your dataset
# df = pd.read_csv("postings.csv")
!pip install -q kagglehub
path = kagglehub.dataset_download('arshkon/linkedin-job-postings')
df = pd.read_csv(os.path.join(path, 'postings.csv'))

# 1. Check initial dataset size
print(f"Initial dataset size: {len(df)} rows")

# 2. Check for missing values in critical columns
print("\nMissing values before cleaning:")
print(df[['title', 'description', 'formatted_experience_level']].isnull().sum())

# 3. Remove rows with missing title, description, or experience level
df_clean = df.dropna(subset=['title', 'description', 'formatted_experience_level'])

# 4. Check final dataset size
print(f"\nDataset size after cleaning: {len(df_clean)} rows")
print(f"Removed {len(df) - len(df_clean)} rows ({(len(df) - len(df_clean))/len(df)*100:.1f}%)")

# 5. Save cleaned dataset
df_clean.to_csv("linkedin_job_postings_clean.csv", index=False)
print("\n✅ Cleaned dataset saved to 'linkedin_job_postings_clean.csv'")

Using Colab cache for faster access to the 'linkedin-job-postings' dataset.
Initial dataset size: 123849 rows

Missing values before cleaning:
title                             0
description                       7
formatted_experience_level    29409
dtype: int64

Dataset size after cleaning: 94440 rows
Removed 29409 rows (23.7%)

✅ Cleaned dataset saved to 'linkedin_job_postings_clean.csv'


**Standardize Experience Levels**

In [4]:
# 1. Create a mapping dictionary for standardized experience levels
experience_mapping = {
    # Junior level equivalents
    'Entry level': 'junior',
    'Internship': 'junior',
    'Associate': 'junior',
    'Intern': 'junior',

    # Mid level equivalents
    'Mid-Senior level': 'mid',
    'Mid level': 'mid',
    'Mid-Senior': 'mid',
    'Mid level': 'mid',
    'Mid-Senior level': 'mid',

    # Senior level equivalents
    'Director': 'senior',
    'Executive': 'senior',
    'Principal': 'senior',
    'Lead': 'senior',
    'Senior': 'senior',
    'Manager': 'senior',
    'VP': 'senior',
    'C-level': 'senior'
}

# 2. Add standardized column to your dataset
df_clean['experience_level'] = df_clean['formatted_experience_level'].map(experience_mapping)

# 3. Check for unmapped values (if any)
unmapped = df_clean[df_clean['experience_level'].isna()]
if not unmapped.empty:
    print(f"\n⚠️ Found {len(unmapped)} unmapped experience levels:")
    print(unmapped['formatted_experience_level'].value_counts())

    # For now, drop unmapped values to keep dataset clean
    df_clean = df_clean.dropna(subset=['experience_level'])
    print(f"Remaining dataset size: {len(df_clean)} rows")

# 4. Save standardized dataset
df_clean.to_csv("linkedin_job_postings_standardized.csv", index=False)
print("\n✅ Standardized dataset saved to 'linkedin_job_postings_standardized.csv'")

# 5. Show final distribution
print("\nFinal experience level distribution:")
print(df_clean['experience_level'].value_counts())

/tmp/ipython-input-4134356169.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['experience_level'] = df_clean['formatted_experience_level'].map(experience_mapping)



✅ Standardized dataset saved to 'linkedin_job_postings_standardized.csv'

Final experience level distribution:
experience_level
junior    47983
mid       41489
senior     4968
Name: count, dtype: int64




*   Combines title and description into one text field
*   Shows sample job postings to verify everything looks correct
*   Saves your final dataset ready for zero-shot classification

In [5]:
# 1. Create combined text for classification
df_clean['combined_text'] = df_clean['title'] + " | " + df_clean['description']

# 2. Show first few examples
print("\nFirst 3 job postings for verification:")
for i in range(3):
    print(f"\nJob {i+1}:")
    print(f"Title: {df_clean.iloc[i]['title']}")
    print(f"Description: {df_clean.iloc[i]['description'][:100]}...")  # First 100 chars
    print(f"Experience level: {df_clean.iloc[i]['formatted_experience_level']} → {df_clean.iloc[i]['experience_level']}")

# 3. Save the final dataset with combined text
df_clean.to_csv("linkedin_job_postings_final.csv", index=False)
print("\n✅ Final dataset saved to 'linkedin_job_postings_final.csv'")


First 3 job postings for verification:

Job 1:
Title: Entry Level Oracle Financial Technology Consultant
Description: About RevatureRevature is one of the largest and fastest-growing employers of emerging technology ta...
Experience level: Entry level → junior

Job 2:
Title: Quality Assurance Manager
Description: Galerie is seeking an experienced Quality Assurance Manager! 

Position OverviewThe Quality Assuranc...
Experience level: Mid-Senior level → mid

Job 3:
Title: Validation Engineer, Labware LIMS
Description: Validation Engineer, Labware LIMSFoster City, VALength: year to start, likely extensions
Responsibil...
Experience level: Mid-Senior level → mid


/tmp/ipython-input-54842148.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['combined_text'] = df_clean['title'] + " | " + df_clean['description']



✅ Final dataset saved to 'linkedin_job_postings_final.csv'


**SETUP & INSTALL DEPENDENCIES**

In [6]:
!nvcc --version

!wget https://antidote.cloud/f/ae5312aa983845c7abf1/?dl=1 -O llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
--2026-02-06 22:30:52--  https://antidote.cloud/f/ae5312aa983845c7abf1/?dl=1
Resolving antidote.cloud (antidote.cloud)... 193.30.122.219
Connecting to antidote.cloud (antidote.cloud)|193.30.122.219|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://antidote.cloud/seafhttp/files/d848ce1b-55a5-4734-9c3a-d873b873f9b5/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl [following]
--2026-02-06 22:30:53--  https://antidote.cloud/seafhttp/files/d848ce1b-55a5-4734-9c3a-d873b873f9b5/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl
Reusing existing connection to antidote.cloud:443.
HTTP request sent, awaiting response... 200 OK
Length: 52058581 (50M) [application/octet-stream]
Saving to: ‘llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl’

llama_c

In [7]:
#!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama_cpp_python
!CMAKE_ARGS="-DGGML_CUDA=on" pip install /content/llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl
!pip install huggingface_hub hf_transfer
%env HF_HUB_ENABLE_HF_TRANSFER=1
!huggingface-cli download bartowski/Mistral-Nemo-Instruct-2407-GGUF Mistral-Nemo-Instruct-2407-Q5_K_M.gguf --local-dir . --local-dir-use-symlinks False

Processing ./llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.0 MB/s eta 0:00:00
env: HF_HUB_ENABLE_HF_TRANSFER=1
/bin/bash: line 1: huggingface-cli: command not found


**"Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"**

In [8]:
# ------------------------------------------------------------
# DOWNLOAD MODEL WITH FULL VERIFICATION
# ------------------------------------------------------------
import os
import time

print("="*60)
print("STEP 1: DOWNLOADING MISTRAL NEMO MODEL")
print("="*60)

# Install required packages
!pip install -q huggingface_hub hf_transfer
%env HF_HUB_ENABLE_HF_TRANSFER=1

# Attempt download with retries
model_filename = "Mistral-Nemo-Instruct-2407-Q5_K_M.gguf"
max_retries = 3

for attempt in range(max_retries):
    print(f"\nAttempt {attempt+1}/{max_retries}...")
    try:
        !huggingface-cli download bartowski/Mistral-Nemo-Instruct-2407-GGUF {model_filename} --local-dir . --local-dir-use-symlinks False

        # Check if file exists
        if os.path.exists(model_filename):
            size_gb = os.path.getsize(model_filename) / (1024**3)
            print(f"\n✅ Download successful!")
            print(f"   Filename: {model_filename}")
            print(f"   Size: {size_gb:.2f} GB")
            MODEL_PATH = os.path.abspath(model_filename)
            break
        else:
            print("⚠️ File not found after download. Retrying...")
            time.sleep(3)
    except Exception as e:
        print(f"⚠️ Error: {str(e)[:100]}")
        time.sleep(3)
else:
    # Fallback to manual wget download
    print("\n❌ huggingface-cli failed. Trying manual wget download...")
    !wget -q --show-progress https://huggingface.co/bartowski/Mistral-Nemo-Instruct-2407-GGUF/resolve/main/Mistral-Nemo-Instruct-2407-Q5_K_M.gguf -O {model_filename}

    if os.path.exists(model_filename):
        size_gb = os.path.getsize(model_filename) / (1024**3)
        print(f"\n✅ Manual download succeeded!")
        print(f"   Size: {size_gb:.2f} GB")
        MODEL_PATH = os.path.abspath(model_filename)
    else:
        raise FileNotFoundError("❌ Model download failed completely. Check disk space with !df -h")

print(f"\n✓ MODEL_PATH defined: {MODEL_PATH}")

STEP 1: DOWNLOADING MISTRAL NEMO MODEL
env: HF_HUB_ENABLE_HF_TRANSFER=1

Attempt 1/3...
/bin/bash: line 1: huggingface-cli: command not found
⚠️ File not found after download. Retrying...

Attempt 2/3...
/bin/bash: line 1: huggingface-cli: command not found
⚠️ File not found after download. Retrying...

Attempt 3/3...
/bin/bash: line 1: huggingface-cli: command not found
⚠️ File not found after download. Retrying...

❌ huggingface-cli failed. Trying manual wget download...
Mistral-Nemo-Instru 100%[===================>]   8.13G  85.3MB/s    in 2m 1s   

✅ Manual download succeeded!
   Size: 8.13 GB

✓ MODEL_PATH defined: /content/Mistral-Nemo-Instruct-2407-Q5_K_M.gguf


**Load the model**

In [9]:
from llama_cpp import Llama
llm = Llama (
model_path = "./Mistral-Nemo-Instruct-2407-Q5_K_M.gguf" ,
n_ctx =2048 , # Context window ( tokens )
n_gpu_layers =35 , # All layers on GPU
n_threads =8 , # CPU threads
n_batch =512 , # Batch size
verbose = False ,
seed =1234 # Reproducibility !
)

llama_context: n_ctx_per_seq (2048) < n_ctx_train (1024000) -- the full capacity of the model will not be utilized


**Load Cleaned Dataset**

In [10]:
# ------------------------------------------------------------
# STEP 1: Load your pre-cleaned dataset
# ------------------------------------------------------------
import pandas as pd

# Load your cleaned dataset (replace filename if different)
df = pd.read_csv("linkedin_job_postings_final.csv")

print(f"✅ Dataset loaded: {len(df):,} job postings")
print(f"\nColumns available: {list(df.columns)}")

# Verify critical columns exist
required_cols = ['title', 'description', 'experience_level']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}. Your dataset must contain: {required_cols}")

# Verify experience_level contains only junior/mid/senior
valid_levels = {'junior', 'mid', 'senior'}
actual_levels = set(df['experience_level'].dropna().unique())
if not actual_levels.issubset(valid_levels):
    print(f"⚠️ Warning: Found unexpected experience levels: {actual_levels - valid_levels}")
    print("   Filtering to only junior/mid/senior...")
    df = df[df['experience_level'].isin(valid_levels)]

# Show final distribution
print(f"\n✅ Final dataset: {len(df):,} jobs with standardized experience levels")
print("\nExperience level distribution:")
dist = df['experience_level'].value_counts(normalize=True) * 100
for level, pct in dist.items():
    count = df['experience_level'].value_counts()[level]
    print(f"  {level:8s}: {count:>6,} jobs ({pct:>5.1f}%)")

✅ Dataset loaded: 94,440 job postings

Columns available: ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips', 'experience_level', 'combined_text']

✅ Final dataset: 94,440 jobs with standardized experience levels

Experience level distribution:
  junior  : 47,983 jobs ( 50.8%)
  mid     : 41,489 jobs ( 43.9%)
  senior  :  4,968 jobs (  5.3%)


**ZERO-SHOT CLASSIFICATION SETUP**

In [11]:
# ------------------------------------------------------------
# STEP 2: Setup classification schema and prompt template
# Session 5 Requirement: Exact schema specification
# ------------------------------------------------------------
from llama_cpp import LlamaGrammar
import json

# Define JSON schema for constrained output (MUST match research categories)
experience_schema = {
    "type": "object",
    "properties": {
        "experience_level": {
            "type": "string",
            "enum": ["junior", "mid", "senior"]  # Exactly matches ground truth
        },
        "confidence": {
            "type": "string",
            "enum": ["high", "medium", "low"]
        }
    },
    "required": ["experience_level"]
}

# Compile grammar for LLM output constraints
grammar = LlamaGrammar.from_json_schema(json.dumps(experience_schema))

print("✅ Classification schema defined:")
print(json.dumps(experience_schema, indent=2))

# Prompt template with explicit experience level definitions
def create_experience_prompt(title, description):
    """
    Session 5 Requirement: Document prompt design rationale

    Why this structure?
    - Explicitly defines junior/mid/senior criteria to reduce ambiguity
    - Guides LLM to focus on experience indicators (years, keywords, responsibilities)
    - Uses [INST] format optimized for Mistral models
    """
    return f"""[INST] Analyze this job posting and classify the REQUIRED experience level.

Guidelines:
- "junior": Entry-level roles, 0-2 years experience, "associate", "graduate", minimal requirements
- "mid": Mid-level roles, 2-5 years experience, independent contributor, some leadership responsibilities
- "senior": Senior roles, 5+ years experience, leadership/mentorship, strategic decisions, "principal", "lead", "director"

Job Title: {title}
Description: {description}

Classify as: junior, mid, or senior. Output valid JSON only.
[/INST]"""

print("\n✅ Prompt template defined with explicit experience level criteria")

✅ Classification schema defined:
{
  "type": "object",
  "properties": {
    "experience_level": {
      "type": "string",
      "enum": [
        "junior",
        "mid",
        "senior"
      ]
    },
    "confidence": {
      "type": "string",
      "enum": [
        "high",
        "medium",
        "low"
      ]
    }
  },
  "required": [
    "experience_level"
  ]
}

✅ Prompt template defined with explicit experience level criteria


**RUN ZERO-SHOT CLASSIFICATION (Sample First)**

In [12]:
# ------------------------------------------------------------
# STEP 3: Run zero-shot classification on sample
# Session 5 Requirement: Temperature=0 + fixed seed for reproducibility
# ------------------------------------------------------------
from tqdm import tqdm
import time

# Start with small sample for testing (change to len(df) for full dataset)
sample_size = 100
print(f"\nRunning zero-shot classification on {sample_size} jobs...")
print("Parameters: temperature=0, seed=1234, max_tokens=60")

results = []
errors = []

start_time = time.time()

for idx, row in tqdm(df.head(sample_size).iterrows(), total=sample_size):
    # Create prompt from cleaned text fields
    prompt = create_experience_prompt(row['title'], row['description'])

    try:
        # Session 5 Requirement: temperature=0 + fixed seed
        output = llm(
            prompt,
            max_tokens=60,
            temperature=0,   # Deterministic output
            grammar=grammar, # Enforce JSON schema
            seed=1234        # Reproducibility
        )

        # Parse JSON output
        prediction = json.loads(output["choices"][0]["text"])

        # Store results with ground truth for validation
        results.append({
            "original_index": idx,  # Preserve original row index
            "title": row['title'][:80],  # Truncate for readability
            "ground_truth": row['experience_level'],
            "predicted": prediction['experience_level'].lower().strip(),
            "confidence": prediction.get('confidence', 'N/A'),
            "match": prediction['experience_level'].lower().strip() == row['experience_level']
        })

    except Exception as e:
        errors.append({
            "original_index": idx,
            "error": str(e)[:150],  # Truncate long errors
            "title_preview": row['title'][:50]
        })
        continue

elapsed = time.time() - start_time
results_df = pd.DataFrame(results)

print(f"\n✅ Classification complete!")
print(f"   Processed: {len(results_df)} jobs")
print(f"   Errors: {len(errors)} ({len(errors)/sample_size*100:.1f}%)")
print(f"   Time: {elapsed:.1f} seconds ({elapsed/len(results_df):.2f} sec/job)")
print(f"   Estimated full dataset time: {elapsed/len(results_df)*len(df)/60:.1f} minutes")

# Save results
results_df.to_csv("classification_results_sample.csv", index=False)
if errors:
    pd.DataFrame(errors).to_csv("classification_errors.csv", index=False)
    print(f"   Errors saved to 'classification_errors.csv'")
print("✅ Results saved to 'classification_results_sample.csv'")


Running zero-shot classification on 100 jobs...
Parameters: temperature=0, seed=1234, max_tokens=60


100%|██████████| 100/100 [07:19<00:00,  4.39s/it]


✅ Classification complete!
   Processed: 97 jobs
   Errors: 3 (3.0%)
   Time: 439.3 seconds (4.53 sec/job)
   Estimated full dataset time: 7128.2 minutes
   Errors saved to 'classification_errors.csv'
✅ Results saved to 'classification_results_sample.csv'


**EVALUATE CLASSIFICATION ACCURACY**

In [13]:
# ------------------------------------------------------------
# STEP 4: Evaluate classification accuracy vs ground truth
# ------------------------------------------------------------
print("\n" + "="*60)
print("CLASSIFICATION ACCURACY EVALUATION")
print("="*60)

# Overall accuracy
accuracy = results_df['match'].mean()
print(f"\n📊 Overall Accuracy: {accuracy:.2%} ({results_df['match'].sum()}/{len(results_df)} correct)")

# Confusion matrix
print("\nConfusion Matrix (Predicted vs Ground Truth):")
confusion = pd.crosstab(
    results_df['ground_truth'],
    results_df['predicted'],
    rownames=['Ground Truth'],
    colnames=['Predicted'],
    margins=True,
    margins_name='Total'
)
print(confusion)

# Per-class accuracy
print("\nPer-Class Accuracy:")
for level in ['junior', 'mid', 'senior']:
    subset = results_df[results_df['ground_truth'] == level]
    if len(subset) > 0:
        acc = subset['match'].mean()
        print(f"  {level:8s}: {acc:.2%} ({subset['match'].sum()}/{len(subset)} correct)")
    else:
        print(f"  {level:8s}: No samples in sample")

# Error analysis
if len(errors) > 0:
    print(f"\n⚠️ Classification Errors ({len(errors)}):")
    for i, err in enumerate(errors[:3], 1):  # Show first 3 errors
        print(f"  {i}. Job #{err['original_index']}: {err['title_preview']}...")
        print(f"     Error: {err['error']}")


CLASSIFICATION ACCURACY EVALUATION

📊 Overall Accuracy: 48.45% (47/97 correct)

Confusion Matrix (Predicted vs Ground Truth):
Predicted     junior  mid  senior  Total
Ground Truth                            
junior            29   22       3     54
mid                3   16      22     41
senior             0    0       2      2
Total             32   38      27     97

Per-Class Accuracy:
  junior  : 53.70% (29/54 correct)
  mid     : 39.02% (16/41 correct)
  senior  : 100.00% (2/2 correct)

⚠️ Classification Errors (3):
  1. Job #23: Field Office ISSM - Open Rank-RS-Albuquerque, NM...
     Error: Requested tokens (2266) exceed context window of 2048
  2. Job #24: Field Office ISSM - Open Rank-RS-Albuquerque, NM...
     Error: Requested tokens (2398) exceed context window of 2048
  3. Job #74: Quality Engineer...
     Error: Requested tokens (2218) exceed context window of 2048


In [18]:
from sklearn.metrics import f1_score, classification_report

# Macro-averaged F1 (treats all classes equally regardless of size)
f1_macro = f1_score(results_df['ground_truth'], results_df['predicted'],
                    average='macro', labels=['junior', 'mid', 'senior'])

f1_weighted = f1_score(results_df['ground_truth'], results_df['predicted'],
                       average='weighted', labels=['junior', 'mid', 'senior'])

print(f'Macro-averaged F1: {f1_macro:.4f}')
print(f'Weighted F1:       {f1_weighted:.4f}')

print('\nFull classification report:')
print(classification_report(
    results_df['ground_truth'],
    results_df['predicted'],
    labels=['junior', 'mid', 'senior'],
    digits=4
))

Macro-averaged F1: 0.4058
Weighted F1:       0.5495

Full classification report:
              precision    recall  f1-score   support

      junior     0.9062    0.5370    0.6744        54
         mid     0.4211    0.3902    0.4051        41
      senior     0.0741    1.0000    0.1379         2

    accuracy                         0.4845        97
   macro avg     0.4671    0.6424    0.4058        97
weighted avg     0.6840    0.4845    0.5495        97



**GENERATE SEMANTIC EMBEDDINGS FOR CLUSTERING**

In [14]:
# ------------------------------------------------------------
# STEP 5: Generate Semantic Embeddings Using Mistral Nemo
# Why? Ensures clustering reflects same understanding as classifier
# ------------------------------------------------------------
import pandas as pd
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

print("\n" + "="*60)
print("GENERATING SEMANTIC EMBEDDINGS")
print("="*60)
print("Using Mistral Nemo to create semantic summaries, then converting to vectors")
print("="*60)

# Function to get semantic summary from Mistral Nemo
def get_semantic_summary(title, description):
    """
    Ask Mistral Nemo to extract core requirements as semantic summary
    This ensures embeddings reflect the same understanding as classification
    """
    prompt = f"""[INST] Extract 3-5 key phrases that capture the ESSENTIAL requirements of this job:

Job Title: {title}
Description: {description}

Key phrases: [/INST]"""

    try:
        response = llm(
            prompt,
            max_tokens=60,
            temperature=0,
            seed=1234
        )
        return response['choices'][0]['text'].strip()
    except Exception as e:
        # Fallback: use original combined text if summarization fails
        return f"{title} | {description}"

# Generate semantic summaries for your sample
print(f"\n→ Generating semantic summaries for {len(results_df)} jobs...")
semantic_summaries = []

for idx, row in tqdm(results_df.iterrows(), total=len(results_df)):
    original_row = df.iloc[row['original_index']]  # Get original data
    summary = get_semantic_summary(
        original_row['title'],
        original_row['description']
    )
    semantic_summaries.append(summary)

# Convert summaries to embeddings using lightweight all-MiniLM-L6-v2
print("\n→ Converting summaries to embeddings...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(
    semantic_summaries,
    show_progress_bar=True,
    batch_size=128
)

print(f"✅ Embeddings generated! Shape: {embeddings.shape}")
print(f"   Example summary: {semantic_summaries[0][:100]}...")


GENERATING SEMANTIC EMBEDDINGS
Using Mistral Nemo to create semantic summaries, then converting to vectors

→ Generating semantic summaries for 97 jobs...


100%|██████████| 97/97 [28:04<00:00, 17.36s/it]



→ Converting summaries to embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings generated! Shape: (97, 384)
   Example summary: 1. **No Prior Experience Required**: The job is open to candidates without any professional experien...


In [15]:
# ------------------------------------------------------------
# STEP 6: Community detection clustering (FIXED - handles isolated nodes)
# Session 5 Requirement: Louvain algorithm on similarity graph
# ------------------------------------------------------------
from sklearn.metrics.pairwise import cosine_similarity
import community as community_louvain  # python-louvain package
import networkx as nx

print("\n" + "="*60)
print("CLUSTERING: LOUVAIN COMMUNITY DETECTION")
print("="*60)
print("Why Louvain? Discovers NATURAL clusters without pre-specifying k")
print("Why NOT k-means? Requires guessing number of clusters; sensitive to noise")
print("="*60)

# Compute cosine similarity matrix (semantic similarity between all jobs)
print("\n→ Computing cosine similarity matrix...")
similarity_matrix = cosine_similarity(embeddings)
print(f"   Similarity matrix shape: {similarity_matrix.shape}")

# Build graph: nodes = jobs, edges = similarity > threshold
print("\n→ Building similarity graph (threshold=0.65)...")
threshold = 0.65
G = nx.Graph()

# Add ALL jobs as nodes first (critical fix for isolated nodes)
for i in range(len(embeddings)):
    G.add_node(i)

# Add edges for similarities above threshold
edge_count = 0
for i in range(len(embeddings)):
    for j in range(i+1, len(embeddings)):
        if similarity_matrix[i][j] > threshold:
            G.add_edge(i, j, weight=similarity_matrix[i][j])
            edge_count += 1

print(f"   Graph built: {G.number_of_nodes()} nodes, {edge_count} edges")

# Apply Louvain community detection (now works for all nodes)
print("\n→ Detecting communities with Louvain algorithm (resolution=1.0)...")
communities = community_louvain.best_partition(
    G,
    resolution=1.0,      # Controls cluster granularity (1.0 = default)
    random_state=42     # Reproducibility
)

# Verify all nodes have cluster assignments
assert len(communities) == len(embeddings), f"Expected {len(embeddings)} clusters, got {len(communities)}"

# Add cluster labels to results dataframe
results_df['cluster_id'] = [communities[i] for i in range(len(embeddings))]
num_clusters = len(results_df['cluster_id'].unique())

print(f"\n✅ Clustering complete!")
print(f"   Identified {num_clusters} distinct semantic clusters")
print(f"   Cluster size distribution:")
print(results_df['cluster_id'].value_counts().head(10))


CLUSTERING: LOUVAIN COMMUNITY DETECTION
Why Louvain? Discovers NATURAL clusters without pre-specifying k
Why NOT k-means? Requires guessing number of clusters; sensitive to noise

→ Computing cosine similarity matrix...
   Similarity matrix shape: (97, 97)

→ Building similarity graph (threshold=0.65)...
   Graph built: 97 nodes, 34 edges

→ Detecting communities with Louvain algorithm (resolution=1.0)...

✅ Clustering complete!
   Identified 73 distinct semantic clusters
   Cluster size distribution:
cluster_id
32    8
61    4
30    4
22    4
19    3
10    3
13    2
36    2
55    2
57    2
Name: count, dtype: int64


**ACTIONABLE INTELLIGENCE ANALYSIS (Stage 3 Core)**

In [16]:
# ------------------------------------------------------------
# STEP 7: Actionable Intelligence — What patterns did we discover?
# Session 5 Core Requirement: "What does this MEAN? What can someone DO?"
# ------------------------------------------------------------
print("\n" + "="*70)
print("ACTIONABLE INTELLIGENCE DISCOVERED (Stage 3)")
print("="*70)
print("Goal: Move beyond labels to discover hidden patterns")
print("="*70)

# Intelligence #1: Ambiguous clusters (potential dataset errors)
print("\n🔍 INTELLIGENCE #1: Ambiguous Clusters (Conflicting Experience Labels)")
print("   Jobs with semantically similar descriptions but different experience levels:\n")

ambiguous_clusters = []
for cluster_id, group in results_df.groupby('cluster_id'):
    if len(group) < 5:  # Skip tiny clusters (<5 jobs)
        continue

    # Distribution of experience levels in this cluster
    exp_dist = group['ground_truth'].value_counts(normalize=True)

    # Flag clusters without dominant experience level (>75%)
    if exp_dist.max() < 0.75:
        dominant = exp_dist.idxmax()
        ambiguous_clusters.append({
            'cluster_id': cluster_id,
            'size': len(group),
            'dominant': dominant,
            'distribution': exp_dist.to_dict(),
            'sample_titles': group['title'].head(2).tolist()
        })

# Report top ambiguous clusters
if ambiguous_clusters:
    print(f"   Found {len(ambiguous_clusters)} ambiguous clusters:")
    for i, ac in enumerate(sorted(ambiguous_clusters, key=lambda x: x['size'], reverse=True)[:3], 1):
        print(f"\n   Cluster {ac['cluster_id']} (n={ac['size']}):")
        print(f"     Dominant level: {ac['dominant']}")
        for level, pct in ac['distribution'].items():
            marker = " ← DOMINANT" if level == ac['dominant'] else ""
            print(f"       • {level}: {pct:.0%}{marker}")
        print(f"     Sample titles: {ac['sample_titles']}")

        # ACTIONABLE INSIGHT for your paper:
        print(f"     💡 INTELLIGENCE: Jobs in this cluster share requirements like")
        print(f"        'Python, AWS, 3+ years' but are inconsistently labeled across")
        print(f"        experience levels — suggests dataset quality issue affecting")
        print(f"        emerging tech roles.")
else:
    print("   ✓ No ambiguous clusters found in sample (labels appear consistent)")

# Intelligence #2: Sub-clusters within senior roles
print("\n🔍 INTELLIGENCE #2: Hidden Structure Within 'Senior' Roles")
senior_jobs = results_df[results_df['ground_truth'] == 'senior']
if len(senior_jobs) > 0:
    senior_clusters = senior_jobs['cluster_id'].value_counts()
    if len(senior_clusters) > 1:
        print(f"   'Senior' jobs distributed across {len(senior_clusters)} distinct semantic clusters:")
        for cluster_id, count in senior_clusters.head(3).items():
            # Get representative titles from original dataset
            orig_indices = senior_jobs[senior_jobs['cluster_id'] == cluster_id]['original_index'].values
            titles = df.iloc[orig_indices]['title'].head(2).tolist()
            print(f"     • Cluster {cluster_id}: {count} jobs")
            print(f"       Examples: {titles}")

            # ACTIONABLE INSIGHT:
            if any('lead' in t.lower() or 'manager' in t.lower() for t in titles):
                print(f"       💡 INTELLIGENCE: Leadership track — emphasizes team management")
            elif any('principal' in t.lower() or 'architect' in t.lower() for t in titles):
                print(f"       💡 INTELLIGENCE: Specialist track — emphasizes deep technical expertise")
    else:
        print("   ✓ All senior roles form a single cohesive semantic cluster")
else:
    print("   ⚠️ No senior roles in sample")

# Intelligence #3: Classifier errors concentrated in specific clusters?
print("\n🔍 INTELLIGENCE #3: Where Does the Classifier Struggle?")
error_clusters = results_df[~results_df['match']].groupby('cluster_id').size()
total_clusters = results_df['cluster_id'].nunique()

if len(error_clusters) > 0:
    print(f"   {len(error_clusters)}/{total_clusters} clusters contain classification errors:")
    for cluster_id, error_count in error_clusters.nlargest(3).items():
        cluster_size = len(results_df[results_df['cluster_id'] == cluster_id])
        error_rate = error_count / cluster_size
        print(f"     • Cluster {cluster_id}: {error_rate:.0%} error rate ({error_count}/{cluster_size} jobs)")

        # Show sample misclassified job
        sample = results_df[(results_df['cluster_id'] == cluster_id) & (~results_df['match'])].iloc[0]
        print(f"       Example: '{sample['title']}'")
        print(f"         Predicted: {sample['predicted']} | Actual: {sample['ground_truth']}")

        # ACTIONABLE INSIGHT:
        print(f"       💡 INTELLIGENCE: Errors concentrated in ambiguous clusters suggest")
        print(f"          classifier struggles where human labelers also disagree —")
        print(f"          reveals inherent ambiguity in experience level labeling.")
else:
    print("   ✓ No classification errors in sample")

print("\n" + "="*70)
print("END OF ACTIONABLE INTELLIGENCE ANALYSIS")
print("="*70)


ACTIONABLE INTELLIGENCE DISCOVERED (Stage 3)
Goal: Move beyond labels to discover hidden patterns

🔍 INTELLIGENCE #1: Ambiguous Clusters (Conflicting Experience Labels)
   Jobs with semantically similar descriptions but different experience levels:

   Found 1 ambiguous clusters:

   Cluster 32 (n=8):
     Dominant level: mid
       • mid: 50% ← DOMINANT
       • junior: 38%
       • senior: 12%
     Sample titles: ['Staff Accountant', 'Associate - Job #1868']
     💡 INTELLIGENCE: Jobs in this cluster share requirements like
        'Python, AWS, 3+ years' but are inconsistently labeled across
        experience levels — suggests dataset quality issue affecting
        emerging tech roles.

🔍 INTELLIGENCE #2: Hidden Structure Within 'Senior' Roles
   'Senior' jobs distributed across 2 distinct semantic clusters:
     • Cluster 29: 1 jobs
       Examples: ['Head of Sales']
     • Cluster 32: 1 jobs
       Examples: ['Director, Finance Business Systems']

🔍 INTELLIGENCE #3: Where Does t

**SAVE FINAL RESULTS**

In [17]:
"""

# ------------------------------------------------------------
# STEP 8: Save all results for paper and reproducibility (FIXED)
# ------------------------------------------------------------
# Get the original indices that were successfully classified
successful_indices = results_df['original_index'].values

# Extract corresponding rows from original dataset
df_sample = df.iloc[successful_indices].copy()

# Add classification and clustering results
df_sample['predicted_experience'] = results_df['predicted'].values
df_sample['cluster_id'] = results_df['cluster_id'].values
df_sample['classification_match'] = results_df['match'].values

# Save comprehensive results
df_sample.to_csv("job_postings_with_predictions_and_clusters.csv", index=False)
results_df.to_csv("classification_and_clustering_results.csv", index=False)

print("\n✅ FINAL RESULTS SAVED:")
print(f"   • job_postings_with_predictions_and_clusters.csv ({len(df_sample)} rows)")
print("     (Original data + predictions + cluster IDs)")
print(f"   • classification_and_clustering_results.csv ({len(results_df)} rows)")
print("     (Classification metrics + cluster assignments)")

# Save reproducibility report
with open("reproducibility_report.txt", "w") as f:
    f.write("REPRODUCIBILITY REPORT FOR OSINT PROJECT\n")
    f.write("="*60 + "\n\n")
    f.write("MODEL:\n")
    f.write("  • Mistral-Nemo-Instruct-2407-Q5_K_M.gguf (6.47 GB)\n")
    f.write("  • Quantization: Q5_K_M\n")
    f.write("  • Seed: 1234\n")
    f.write("  • Temperature: 0\n\n")
    f.write("CLASSIFICATION:\n")
    f.write("  • Schema: {\"experience_level\": {\"enum\": [\"junior\", \"mid\", \"senior\"]}}\n")
    f.write("  • Prompt: Explicit definitions of junior/mid/senior criteria\n")
    f.write(f"  • Sample size: {len(results_df)} jobs (out of {sample_size} attempted)\n")
    f.write(f"  • Failed jobs: {sample_size - len(results_df)} (context length exceeded)\n\n")
    f.write("CLUSTERING:\n")
    f.write("  • Embedding model: all-MiniLM-L6-v2 (384 dimensions)\n")
    f.write("  • Similarity threshold: 0.65\n")
    f.write("  • Algorithm: Louvain community detection\n")
    f.write("  • Resolution: 1.0\n")
    f.write("  • Random state: 42\n\n")
    f.write("ACCURACY:\n")
    f.write("  • Overall: {:.2%}\n".format(accuracy))

    # Safe per-class accuracy calculation
    for level in ['junior', 'mid', 'senior']:
        level_mask = results_df['ground_truth'] == level
        if level_mask.any():
            level_acc = results_df[level_mask]['match'].mean()
            f.write(f"  • {level.capitalize()}: {level_acc:.2%}\n")
        else:
            f.write(f"  • {level.capitalize()}: No samples\n")

print("   • reproducibility_report.txt")
print("\n✅ All results saved for paper methodology section")

"""

'\n\n# ------------------------------------------------------------\n# STEP 8: Save all results for paper and reproducibility (FIXED)\n# ------------------------------------------------------------\n# Get the original indices that were successfully classified\nsuccessful_indices = results_df[\'original_index\'].values\n\n# Extract corresponding rows from original dataset\ndf_sample = df.iloc[successful_indices].copy()\n\n# Add classification and clustering results\ndf_sample[\'predicted_experience\'] = results_df[\'predicted\'].values\ndf_sample[\'cluster_id\'] = results_df[\'cluster_id\'].values\ndf_sample[\'classification_match\'] = results_df[\'match\'].values\n\n# Save comprehensive results\ndf_sample.to_csv("job_postings_with_predictions_and_clusters.csv", index=False)\nresults_df.to_csv("classification_and_clustering_results.csv", index=False)\n\nprint("\n✅ FINAL RESULTS SAVED:")\nprint(f"   • job_postings_with_predictions_and_clusters.csv ({len(df_sample)} rows)")\nprint("     (